In [ ]:
"""
Bootstrap cluster-stability analysis.
No patient data are included; set the input matrix X to the local data.
"""

import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering
from sklearn.metrics import adjusted_rand_score

LATENT_DIM = 20


def compute_pca_features(Xarr):
    Xarr = np.asarray(Xarr, dtype=np.float32)
    input_dim = Xarr.shape[1]

    autoencoder = train_autoencoder(input_dim, LATENT_DIM, Xarr)
    decoded = autoencoder.predict(Xarr, verbose=0)
    rec_error = np.mean((Xarr - decoded) ** 2, axis=0)

    sorted_indices = np.argsort(rec_error)[::-1]
    mediana = np.median(rec_error)
    Q1, Q3 = np.percentile(rec_error, [25, 75])
    IQR = Q3 - Q1
    lower, upper = mediana - 3 * IQR, mediana + 3 * IQR

    top = [i for i in sorted_indices if lower <= rec_error[i] <= upper]
    if len(top) < 5:
        top = list(sorted_indices[:5])
    elif len(top) > 20:
        top = list(sorted_indices[:20])

    selected = Xarr[:, top]
    reduced_dim = min(LATENT_DIM, selected.shape[1])
    encoded = PCA(n_components=reduced_dim, svd_solver="full",
                  random_state=42).fit_transform(selected)
    return encoded


def _cluster(encoded, method, k=2):
    n_neighbors = max(2, min(10, len(encoded) - 1))
    if method == "kmeans":
        model = KMeans(n_clusters=k, random_state=42, n_init=50)
    elif method == "agglomerative":
        model = AgglomerativeClustering(n_clusters=k)
    elif method == "spectral":
        model = SpectralClustering(n_clusters=k, random_state=42, n_init=50,
                                   affinity="nearest_neighbors",
                                   n_neighbors=n_neighbors,
                                   assign_labels="kmeans",
                                   eigen_solver="arpack")
    else:
        raise ValueError(method)
    return model.fit_predict(encoded)


def bootstrap_stability_clustering_only(encoded, method, k=2, B=1000, seed=42):
    encoded = np.asarray(encoded, dtype=np.float32)
    n = encoded.shape[0]

    ref_labels = _cluster(encoded, method, k)

    rng = np.random.RandomState(seed)
    aris = []
    for b in range(B):
        pos = rng.randint(0, n, size=n)
        try:
            boot_labels = _cluster(encoded[pos], method, k)
        except Exception as e:
            print(f"  [skip b={b}] {type(e).__name__}: {e}")
            continue

        boot_by_row = {}
        for row_id, lab in zip(pos, boot_labels):
            if row_id not in boot_by_row:
                boot_by_row[row_id] = lab

        ids = list(boot_by_row.keys())
        ref_vec = [ref_labels[i] for i in ids]
        boot_vec = [boot_by_row[i] for i in ids]
        aris.append(adjusted_rand_score(ref_vec, boot_vec))

    aris = np.array(aris)
    return {"method": method, "k": k, "B_done": len(aris),
            "ari_mean": float(aris.mean()), "ari_std": float(aris.std()),
            "ari_median": float(np.median(aris)),
            "ari_q1": float(np.percentile(aris, 25)),
            "ari_q3": float(np.percentile(aris, 75)),
            "ari_all": aris}


if __name__ == "__main__":
    # X = feature matrix (already z-scored) for one diffusion model / group.
    set_seed(42)
    encoded = compute_pca_features(X)

    B = 1000
    for method in ("kmeans", "agglomerative", "spectral"):
        res = bootstrap_stability_clustering_only(encoded, method, k=2, B=B)
        lo, hi = np.percentile(res["ari_all"], [2.5, 97.5])
        print(f"{method:14s}  median={res['ari_median']:.3f} "
              f"[95% CI {lo:.3f}, {hi:.3f}]  (B={res['B_done']})")